In [125]:
import numpy as np
import pandas as pd
import pyxdf
import mne
import matplotlib.pyplot as plt
from pathlib import Path
from mne.time_frequency import stft, tfr_morlet

## Induced power (red/blue) TFR, combined across all 4 runs, per participant

Everything below is self-contained (doesn't rely on the single-file demo above). For
each participant it loads all 4 runs and locates the real `LIFU_ON` stimulation markers.
Firing is confirmed by eye rather than auto-detected: the cells below plot every trial's
raw STFT spectrogram (same check as `new_electric_artifacts.ipynb`), and you list which
trial indices to keep in `FIRED_TRIAL_INDICES` -- the same manual approach as
`trial_index_df.drop(...)` above, just applied per run across all 4 runs.


In [126]:
XDF_ROOT = Path(r"C:\Users\jshin\OW_closedloopLIFU\xdf_data")
PARTICIPANTS = ["dave", "scott"]
RUNS = [1, 2, 3, 4]
CH_NAMES = ["FCz", "CP3", "P5", "Cz", "Pz", "POz", "CP4"]
DATA_COLS = ["Ch0", "Ch1", "Ch2", "Ch3", "Ch4", "Ch5", "Ch6"]  # maps 1:1 to CH_NAMES

def xdf_path_for(participant, run):
    folder = f"sub-{participant}_run_{run}"
    fname = f"sub-{participant}_run_{run}_ses-1_task-{participant}_run_{run}_run-001_eeg.xdf"
    return XDF_ROOT / folder / "ses-1" / "eeg" / fname

In [ ]:
def _load_eeg_and_lifu_markers(xdf_path):
    """Load one XDF recording; return the raw EEG dataframe (with Timestamp) and the
    array of real LIFU_ON stimulation-onset timestamps (empty array if the run has none)."""
    data, _ = pyxdf.load_xdf(str(xdf_path))
    # Streams aren't always in the same order across recordings, so look them up by name.
    streams = {s['info']['name'][0]: s for s in data}

    eeg_stream = streams['EEG_gpype']
    eeg_df = pd.DataFrame(eeg_stream['time_series'])
    eeg_df = eeg_df.rename(columns={i: f"Ch{i}" for i in range(eeg_df.shape[1])})
    eeg_df['Timestamp'] = eeg_stream['time_stamps']
    eeg_raw = eeg_df[DATA_COLS + ['Timestamp']]

    lifu_stream = streams['EEG_LIFU_events']
    lifu_df = pd.DataFrame(lifu_stream['time_series']).rename(columns={0: 'markers'})
    lifu_df['Timestamp'] = lifu_stream['time_stamps']
    lifu_on_ts = np.asarray(lifu_df.loc[lifu_df['markers'] == 'LIFU_ON', 'Timestamp'])

    return eeg_raw, lifu_on_ts


def load_clean_raw_and_lifu_events(xdf_path, sfreq=250):
    """Load one XDF recording, return a cleaned/filtered Raw plus the sample indices
    of every real LIFU_ON stimulation marker (empty array if the run has none)."""
    eeg_raw, lifu_on_ts = _load_eeg_and_lifu_markers(xdf_path)

    timestamps = eeg_raw['Timestamp'].values
    if len(lifu_on_ts) == 0:
        event_samples = np.array([], dtype=int)
    else:
        insert_idx = np.clip(np.searchsorted(timestamps, lifu_on_ts), 1, len(timestamps) - 1)
        left_vals = timestamps[insert_idx - 1]
        right_vals = timestamps[insert_idx]
        use_left = np.abs(lifu_on_ts - left_vals) < np.abs(lifu_on_ts - right_vals)
        event_samples = np.where(use_left, insert_idx - 1, insert_idx)

    # clean eeg
    df = eeg_raw[DATA_COLS].copy()
    df = df.replace([np.inf, -np.inf], np.nan)
    df = df.replace(-200000.0, np.nan)

    # Drop bad channels/ data points
    bad_frac = df.isna().mean()
    bad_channels = list(bad_frac[bad_frac > 0.5].index)

    df = df.interpolate(method='linear', limit=5, limit_direction='both')
    df = df.bfill().ffill()
    df = df[DATA_COLS]

    # MNE object building
    info = mne.create_info(ch_names=CH_NAMES, sfreq=sfreq, ch_types='eeg')
    raw = mne.io.RawArray(df.values.T, info, verbose=False)
    raw.set_montage(mne.channels.make_standard_montage("standard_1020"))
    raw.info['bads'] = [CH_NAMES[DATA_COLS.index(c)] for c in bad_channels]
    if raw.info['bads']:
        print(f"  Bad channel(s) detected: {raw.info['bads']} -> interpolating from neighbors")
        raw.interpolate_bads(reset_bads=True, verbose=False)
    # filtering
    raw.notch_filter(60., verbose=False)
    raw.filter(1., 40., fir_design='firwin', verbose=False)
    raw.set_eeg_reference('average', verbose=False)

    # Fit ICA for inspection only; no components are auto-excluded though
    ica = mne.preprocessing.ICA(n_components=0.99999, random_state=97, max_iter='auto', verbose=False)
    ica.fit(raw, verbose=False)

    return raw, event_samples


def make_epochs(raw, event_samples, tmin, tmax, baseline):
    if len(event_samples) == 0:
        return None
    events = np.column_stack([event_samples,
                               np.zeros_like(event_samples, dtype=int),
                               np.ones_like(event_samples, dtype=int)])
    return mne.Epochs(raw, events, event_id=1, tmin=tmin, tmax=tmax,
                       baseline=baseline, preload=True, verbose=False)


In [128]:
# compute PSDs for epochs that sonicated 
# cross reference electric_artifacts.ipynb to see which ones sonicated, and manually input them

CHECK_PRE, CHECK_POST = 2, 7  # matches new_electric_artifacts.ipynb

FIRED_TRIAL_INDICES = {
    "dave":  {1: [], 2: [0,1,2,3,4], 3: [], 4: [1,6,7,8,9]},
    "scott": {1: [0,2], 2: [], 3: [0], 4: []},
}


def get_check_windows(xdf_path, fs=250):
    """Raw (uncleaned, unfiltered) per-trial EEG windows around each real LIFU_ON marker,
    exactly like new_electric_artifacts.ipynb, plus the matched EEG sample index for each."""
    eeg_raw, lifu_on_ts = _load_eeg_and_lifu_markers(xdf_path)
    timestamps = eeg_raw['Timestamp'].values
    pre_samples = int(CHECK_PRE * fs)
    post_samples = int(CHECK_POST * fs)
    window_len = pre_samples + post_samples

    windows, sample_idxs = [], []
    for event in lifu_on_ts:
        center_idx = int(np.abs(timestamps - event).argmin())
        start_idx = center_idx - pre_samples
        end_idx = start_idx + window_len
        if start_idx < 0 or end_idx > len(eeg_raw):
            continue
        windows.append(eeg_raw.iloc[start_idx:end_idx][DATA_COLS].to_numpy().T)
        sample_idxs.append(center_idx)
    return windows, np.array(sample_idxs, dtype=int)


def compute_epoch_spectrogram(ep_data, sfreq, n_fft=256, step=64):
    """Channel-averaged time-resolved power spectrogram (dB) for one window, (n_channels, n_times) --> (freq, time)."""
    Zxx = stft(ep_data, wsize=n_fft, tstep=step)
    power = np.abs(Zxx) ** 2 / (n_fft ** 2)
    freqs = np.linspace(0, sfreq / 2, power.shape[1])
    return freqs, 10 * np.log10(power.mean(axis=0) + 1e-20)


In [129]:
TFR_TMIN, TFR_TMAX = -5, 20
TFR_BASELINE = (-0.2, 0)

participant_fired_epochs = {}

for participant in PARTICIPANTS:
    keep_indices = FIRED_TRIAL_INDICES[participant]
    fired_samples_by_run = {}

    # plot every trial, sanity check to see which runs are included vs excluded
    for run in RUNS:
        path = xdf_path_for(participant, run)
        windows, sample_idxs = get_check_windows(path)
        if len(windows) == 0:
            print(f"{participant} run {run}: no LIFU_ON events -- skipping.")
            continue

        keep = set(keep_indices.get(run, []))
        fired_samples_by_run[run] = [sample_idxs[i] for i in keep if i < len(sample_idxs)]

        for i, (window, sample_idx) in enumerate(zip(windows, sample_idxs)):
            freqs, db = compute_epoch_spectrogram(window, sfreq=250)
            times = np.linspace(-CHECK_PRE, CHECK_POST, db.shape[-1])
            status = "included" if i in keep else "excluded"

            vmin, vmax = np.percentile(db, [5, 95])
            plt.figure(figsize=(8, 5))
            plt.imshow(db, aspect='auto', origin='lower',
                       extent=[times[0], times[-1], freqs[0], freqs[-1]],
                       vmin=vmin, vmax=vmax, cmap='viridis')
            plt.xlabel("Time (s)")
            plt.ylabel("Frequency (Hz)")
            plt.title(f"{participant} run {run}, trial {i} (sample {sample_idx}) - {status}")
            plt.colorbar(label="Power (dB)")
            plt.show() # sanity check --> remove if you don't want plots

    fired_epoch_list = []
    for run, fired_samples in fired_samples_by_run.items():
        if not fired_samples:
            continue
        path = xdf_path_for(participant, run)
        raw, event_samples = load_clean_raw_and_lifu_events(path)
        main_epochs = make_epochs(raw, event_samples, tmin=TFR_TMIN, tmax=TFR_TMAX, baseline=TFR_BASELINE)
        keep_mask = np.isin(main_epochs.events[:, 0], fired_samples)
        if keep_mask.sum() == 0:
            print(f"  {participant} run {run}: included trials fell too close to the recording edge for the wider TFR window -- skipping.")
            continue
        fired_epoch_list.append(main_epochs[keep_mask])

    if fired_epoch_list:
        participant_fired_epochs[participant] = mne.concatenate_epochs(fired_epoch_list, on_mismatch='warn')
        print(f"\n{participant}: {len(participant_fired_epochs[participant])} included trials combined across runs\n")
    else:
        participant_fired_epochs[participant] = None
        print(f"\n{participant}: no trials included in any run\n")


In [130]:
# Induced Theta/Alpha power change (log-ratio vs. baseline), one red/blue TFR per participant
tfr_freqs = np.linspace(3.0, 40.0, num=38)
tfr_n_cycles = tfr_freqs / 2.0

for participant, epochs in participant_fired_epochs.items():
    if epochs is None or len(epochs) == 0:
        print(f"{participant}: no fired trials available, skipping TFR plot.")
        continue

    power = tfr_morlet(epochs, freqs=tfr_freqs, n_cycles=tfr_n_cycles, use_fft=True,
                        return_itc=False, n_jobs=-1, verbose=False)
    power.crop(tmin=-4, tmax=19)  # trim 1s of wavelet edge effects from each side

    power.plot(
        picks='data',
        baseline=(-2, 0),
        mode='logratio',
        title=f"{participant.capitalize()} Induced Power Changes (fired trials only, N={len(epochs)})",
        combine='mean',
    )